# FunRL-style GRPO Fine-Tuning of Qwen3.5-4B for Tool Calling

Trained a LoRA adapter on Qwen3.5-4B using GRPO (Group Relative Policy Optimization), inspired by the FunRL paper's approach to improving function-calling reliability.

**Setup:** Free Kaggle T4 GPU (single, 15GB VRAM), 4-bit QLoRA via Unsloth.

**Training stages (BFCL dataset):**
1. `simple` — single function, single call (400 examples, 1 epoch)
2. `multiple` — disambiguating between several candidate functions (200 examples, 1 epoch)

**Held-out evaluation (`parallel` category — never trained on, requires multiple simultaneous tool calls):**
- Base Qwen3.5-4B: **0.33** average score
- Fine-tuned adapter: **0.73** average score

**Caveats:** small eval set (15 examples), custom reward/scoring function rather than the official BFCL harness, LoRA adapter not a full fine-tune. Treat this as a promising signal, not a rigorous benchmark result.

**Note on process:** this notebook was built through repeated Kaggle session disconnects and mid-run recoveries — cells reflect real debugging (checkpoint resume logic, tokenizer/processor quirks for the VLM-based Qwen3.5 architecture, thinking-token truncation issues) rather than a clean linear run. Left as-is intentionally.

## Stage 1: Training on `simple` (single function, single call)

In [ ]:
from huggingface_hub import hf_hub_download
import json

q_path = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_simple.json",
    repo_type="dataset",
)
a_path = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="possible_answer/BFCL_v3_simple.json",
    repo_type="dataset",
)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

questions = load_jsonl(q_path)
answers = load_jsonl(a_path)


In [ ]:
gt_by_id = {a["id"]: a["ground_truth"][0] for a in answers}

def build_example(q):
    fn = q["function"][0]
    user_msg = q["question"][0][0]["content"]
    prompt = (
        f"Available function:\n{json.dumps(fn, indent=2)}\n\n"
        f"User: {user_msg}\n"
        f"Call function, wrap in <tool_call></tool_call>, format: "
        f'{{"name": ..., "arguments": {{...}}}}'
    )
    return {"prompt": [{"role": "user", "content": prompt}], "id": q["id"]}

train_data = [build_example(q) for q in questions]
id_to_gt = gt_by_id

from datasets import Dataset
ds = Dataset.from_list(train_data)


In [ ]:
import re, json

def extract_call(text):
    m = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(1).strip())
    except json.JSONDecodeError: return None

def bfcl_reward(completions, id, **kwargs):
    rewards = []
    for comp, ex_id in zip(completions, id):
        call = extract_call(comp[0]["content"])
        gt = id_to_gt[ex_id]
        if call is None:
            rewards.append(-1.0); continue
        fn_name = call.get("name")
        args = call.get("arguments", {})
        if fn_name not in gt:
            rewards.append(-0.5); continue
        gt_params = gt[fn_name]
        score, total = 0, len(gt_params)
        for param, allowed in gt_params.items():
            pred_val = args.get(param, "")
            if pred_val in allowed or str(pred_val) in allowed:
                score += 1
        rewards.append(score / total if total else 0.0)
    return rewards


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048,
    load_in_4bit = True,
    fast_inference = False,   # vLLM doesn't support Qwen3.5 yet
)

if hasattr(tokenizer, "tokenizer") and not hasattr(tokenizer, "pad_token_id"):
    tokenizer.pad_token_id = tokenizer.tokenizer.pad_token_id

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir = "funrl-qwen3.5-4b",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_completion_length = 256,
    beta = 0.02,
    learning_rate = 1e-6,
    max_steps = 400,
    logging_steps = 1,
    save_steps = 25,
    save_total_limit = 2,
    chat_template_kwargs = {"enable_thinking": False},
)

trainer = GRPOTrainer(
    model = model,
    args = config,
    train_dataset = ds,
    reward_funcs = [bfcl_reward],
)
trainer.processing_class.pad_token_id = trainer.processing_class.tokenizer.pad_token_id
trainer.train()


In [ ]:
model.save_pretrained("funrl-qwen3.5-4b-final")
tokenizer.save_pretrained("funrl-qwen3.5-4b-final")


**Result:** reward climbed from the -1 floor (no valid call) to sitting near 1.0 (correct call) for the majority of steps from ~step 100 onward through step 400. Full epoch over 400 `simple` examples.

## Stage 2: Continued training on `multiple` (disambiguating between candidate functions)

Continues fine-tuning the *same* adapter from Stage 1 — not a fresh model.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install --upgrade unsloth unsloth_zoo -q
!pip install --upgrade transformers trl datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 23.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 12.2 MB/s eta 0:00:00
   ━

In [2]:
from huggingface_hub import hf_hub_download
import json

q_path = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_multiple.json",
    repo_type="dataset",
)
a_path = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="possible_answer/BFCL_v3_multiple.json",
    repo_type="dataset",
)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

m_questions = load_jsonl(q_path)
m_answers = load_jsonl(a_path)

print(len(m_questions), len(m_answers))
print(json.dumps(m_questions[0], indent=2))
print(json.dumps(m_answers[0], indent=2))

BFCL_v3_multiple.json: 0.00B [00:00, ?B/s]

BFCL_v3_multiple.json: 0.00B [00:00, ?B/s]

200 200
{
  "id": "multiple_0",
  "question": [
    [
      {
        "role": "user",
        "content": "Can I find the dimensions and properties of a triangle, if I know its three sides are 5 units, 4 units and 3 units long?"
      }
    ]
  ],
  "function": [
    {
      "name": "triangle_properties.get",
      "description": "Retrieve the dimensions, such as area and perimeter, of a triangle if lengths of three sides are given.",
      "parameters": {
        "type": "dict",
        "properties": {
          "side1": {
            "type": "integer",
            "description": "The length of first side of the triangle."
          },
          "side2": {
            "type": "integer",
            "description": "The length of second side of the triangle."
          },
          "side3": {
            "type": "integer",
            "description": "The length of third side of the triangle."
          },
          "get_area": {
            "type": "boolean",
            "description": "

In [3]:
# join prompt builder

import json

def build_example_multiple(q):
    fns = q["function"]  # now a list of multiple schemas
    user_msg = q["question"][0][0]["content"]
    fn_list_str = "\n\n".join(json.dumps(fn, indent=2) for fn in fns)
    prompt = (
        f"Available functions:\n{fn_list_str}\n\n"
        f"User: {user_msg}\n"
        f"Pick the correct function and call it, wrap in <tool_call></tool_call>, format: "
        f"{{\"name\": ..., \"arguments\": {{...}}}}"
    )
    return {"prompt": [{"role": "user", "content": prompt}], "id": q["id"]}

gt_by_id_multiple = {a["id"]: a["ground_truth"][0] for a in m_answers}
train_data_multiple = [build_example_multiple(q) for q in m_questions]

from datasets import Dataset
ds_multiple = Dataset.from_list(train_data_multiple)

In [5]:
#reward fn

import re, json

def extract_call(text):
    m = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(1).strip())
    except json.JSONDecodeError: return None

def bfcl_reward(completions, id, **kwargs):
    rewards = []
    for comp, ex_id in zip(completions, id):
        call = extract_call(comp[0]["content"])
        gt = id_to_gt[ex_id]
        if call is None:
            rewards.append(-1.0); continue
        fn_name = call.get("name")
        args = call.get("arguments", {})
        if fn_name not in gt:
            rewards.append(-0.5); continue
        gt_params = gt[fn_name]
        score, total = 0, len(gt_params)
        for param, allowed in gt_params.items():
            pred_val = args.get(param, "")
            if pred_val in allowed or str(pred_val) in allowed:
                score += 1
        rewards.append(score / total if total else 0.0)
    return rewards

In [ ]:
from datasets import Dataset
ds = Dataset.from_list(train_data)

In [ ]:
print(ds[0])

fake_completion = [{"content": '<tool_call>{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 5, "unit": "units"}}</tool_call>'}]
print(bfcl_reward([fake_completion], id=["simple_0"]))

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048,
    load_in_4bit = True,
    fast_inference = False,
)

if hasattr(tokenizer, "tokenizer") and not hasattr(tokenizer, "pad_token_id"):
    tokenizer.pad_token_id = tokenizer.tokenizer.pad_token_id

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    use_gradient_checkpointing = "unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen3_5 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [17]:
print(trainer.state.global_step)

52


In [12]:
from trl import GRPOConfig, GRPOTrainer

config_multiple = GRPOConfig(
    output_dir = "funrl-qwen3.5-4b-multiple",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_completion_length = 256,
    beta = 0.02,
    learning_rate = 1e-6,
    max_steps = 400,
    logging_steps = 1,
    save_steps = 10,
    save_total_limit = 2,
    chat_template_kwargs = {"enable_thinking": False},
)

id_to_gt = gt_by_id_multiple  # reward fn reads this global

trainer_multiple = GRPOTrainer(
    model = model,
    args = config_multiple,
    train_dataset = ds_multiple,
    reward_funcs = [bfcl_reward],
)
trainer_multiple.processing_class.pad_token_id = trainer_multiple.processing_class.tokenizer.pad_token_id
trainer_multiple.args.max_steps = 200
trainer_multiple.train(resume_from_checkpoint=True)  # resumes from step 25, only needs 55 more

Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / bfcl_reward / mean,rewards / bfcl_reward / std
31,0.001871,1.000000,0.000000,42.250000,38.000000,55.000000,0.000000,42.250000,38.000000,55.000000,0.093543,1.000000,0.000000
32,0.014722,0.500000,1.000000,42.250000,42.000000,43.000000,0.000000,42.250000,42.000000,43.000000,0.736082,0.500000,1.000000
33,0.000901,1.000000,0.000000,37.000000,37.000000,37.000000,0.000000,37.000000,37.000000,37.000000,0.045034,1.000000,0.000000
34,0.000108,1.000000,0.000000,32.000000,32.000000,32.000000,0.000000,32.000000,32.000000,32.000000,0.005383,1.000000,0.000000
35,0.000024,0.833333,0.192450,41.500000,41.000000,42.000000,0.000000,41.500000,41.000000,42.000000,0.001202,0.833333,0.192450
36,0.000516,1.000000,0.000000,44.000000,44.000000,44.000000,0.000000,44.000000,44.000000,44.000000,0.025789,1.000000,0.000000
37,0.000443,0.916667,0.166667,42.500000,41.000000,47.000000,0.000000,42.500000,41.000000,47.000000,0.022130,0.916667,0.166667
38,0.000019,1.000000,0.000000,31.000000,31.000000,31.000000,0.000000,31.000000,31.000000,31.000000,0.000952,1.000000,0.000000
39,0.000471,1.000000,0.000000,44.000000,44.000000,44.000000,0.000000,44.000000,44.000000,44.000000,0.023541,1.000000,0.000000
40,0.000465,1.000000,0.000000,33.000000,33.000000,33.000000,0.000000,33.000000,33.000000,33.000000,0.023239,1.000000,0.000000


TrainOutput(global_step=200, training_loss=0.0006708948943355608, metrics={'train_runtime': 3434.6575, 'train_samples_per_second': 0.233, 'train_steps_per_second': 0.058, 'total_flos': 0.0, 'train_loss': 0.0006708948943355608, 'epoch': 1.0})

In [16]:
import pandas as pd
df = pd.DataFrame(trainer_multiple.state.log_history)
print(df[["step", "reward", "reward_std"]].dropna().to_string())

     step    reward  reward_std
0       1  1.000000    0.000000
1       2  1.000000    0.000000
2       3  0.250000    0.833333
3       4  1.000000    0.000000
4       5  1.000000    0.000000
5       6  1.000000    0.000000
6       7  1.000000    0.000000
7       8  1.000000    0.000000
8       9  1.000000    0.000000
9      10  0.500000    1.000000
10     11  0.500000    1.000000
11     12  1.000000    0.000000
12     13  1.000000    0.000000
13     14  1.000000    0.000000
14     15  1.000000    0.000000
15     16  0.500000    1.000000
16     17  1.000000    0.000000
17     18  1.000000    0.000000
18     19  1.000000    0.000000
19     20  1.000000    0.000000
20     21  1.000000    0.000000
21     22  1.000000    0.000000
22     23  1.000000    0.000000
23     24  1.000000    0.000000
24     25  1.000000    0.000000
25     26  1.000000    0.000000
26     27  0.937500    0.125000
27     28  1.000000    0.000000
28     29  0.875000    0.250000
29     30  1.000000    0.000000
30     3

In [ ]:
print(1)

In [ ]:
inputs = tokenizer.apply_chat_template(
    prompt, tokenize=True, return_tensors="pt",
    add_generation_prompt=True, enable_thinking=False
).to(model.device)
out = model.generate(inputs, max_new_tokens=256)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
import pandas as pd
df = pd.DataFrame(trainer_multiple.state.log_history)
print(df[["step", "reward", "reward_std"]].dropna().to_string())

In [14]:
model.save_pretrained("funrl-qwen3.5-4b-final")
tokenizer.save_pretrained("funrl-qwen3.5-4b-final")

Unsloth: Restored added_tokens_decoder metadata in funrl-qwen3.5-4b-final/tokenizer_config.json.


['funrl-qwen3.5-4b-final/processor_config.json']

In [17]:
model.save_pretrained("funrl-qwen3.5-4b-multiple-final")
tokenizer.save_pretrained("funrl-qwen3.5-4b-multiple-final")

Unsloth: Restored added_tokens_decoder metadata in funrl-qwen3.5-4b-multiple-final/tokenizer_config.json.


['funrl-qwen3.5-4b-multiple-final/processor_config.json']

In [22]:
import os
print(os.listdir("funrl-qwen3.5-4b-final"))

['processor_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'chat_template.jinja']


In [18]:
import shutil
shutil.make_archive("funrl-qwen3.5-4b-multiple-final", 'zip', "funrl-qwen3.5-4b-final")

'/kaggle/working/funrl-qwen3.5-4b-multiple-final.zip'

In [24]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "funrl-qwen3.5-4b-final",  # loads base + adapter if config points to it right
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# test on a new, unseen function-calling prompt

==((====))==  Unsloth 2026.7.6: Fast Qwen3_5 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
                (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
   

In [ ]:
import pandas as pd
df = pd.DataFrame(trainer.state.log_history)
print(df[["step", "reward", "reward_std"]].dropna())

In [26]:
FastLanguageModel.for_inference(model)

test_prompt = [{"role": "user", "content": (
    "Available function:\n"
    '{"name": "calculate_triangle_area", "description": "Calculate the area of a triangle given its base and height.", '
    '"parameters": {"type": "dict", "properties": {"base": {"type": "integer"}, "height": {"type": "integer"}, "unit": {"type": "string"}}, "required": ["base", "height"]}}\n\n'
    "User: What's the area of a triangle with base 7 and height 12?\n"
    'Call function, wrap in <tool_call></tool_call>, format: {"name": ..., "arguments": {...}}'
)}]

inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, return_tensors="pt",
    add_generation_prompt=True, enable_thinking=False
).to(model.device)

out = model.generate(inputs, max_new_tokens=128)
print(tokenizer.decode(out[0], skip_special_tokens=True))

user
Available function:
{"name": "calculate_triangle_area", "description": "Calculate the area of a triangle given its base and height.", "parameters": {"type": "dict", "properties": {"base": {"type": "integer"}, "height": {"type": "integer"}, "unit": {"type": "string"}}, "required": ["base", "height"]}}

User: What's the area of a triangle with base 7 and height 12?
Call function, wrap in <tool_call></tool_call>, format: {"name": ..., "arguments": {...}}
assistant
<think>

</think>

<tool_call>
{"name": "calculate_triangle_area", "arguments": {"base": 7, "height": 12, "unit": "unknown"}}
</tool_call>



In [19]:

# testing session

from huggingface_hub import hf_hub_download
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

q_path = hf_hub_download(repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_parallel.json", repo_type="dataset")
a_path = hf_hub_download(repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="possible_answer/BFCL_v3_parallel.json", repo_type="dataset")

p_questions = load_jsonl(q_path)[:15]  # first 15, unseen
p_answers = load_jsonl(a_path)[:15]
gt_by_id_parallel = {a["id"]: a["ground_truth"] for a in p_answers}

BFCL_v3_parallel.json: 0.00B [00:00, ?B/s]

BFCL_v3_parallel.json: 0.00B [00:00, ?B/s]

In [20]:
print(json.dumps(p_questions[0], indent=2))
print(json.dumps(p_answers[0], indent=2))

{
  "id": "parallel_0",
  "question": [
    [
      {
        "role": "user",
        "content": "Play songs from the artists Taylor Swift and Maroon 5, with a play time of 20 minutes and 15 minutes respectively, on Spotify."
      }
    ]
  ],
  "function": [
    {
      "name": "spotify.play",
      "description": "Play specific tracks from a given artist for a specific time duration.",
      "parameters": {
        "type": "dict",
        "properties": {
          "artist": {
            "type": "string",
            "description": "The artist whose songs you want to play."
          },
          "duration": {
            "type": "integer",
            "description": "The duration for which the songs should be played, in minutes."
          }
        },
        "required": [
          "artist",
          "duration"
        ]
      }
    }
  ]
}
{
  "id": "parallel_0",
  "ground_truth": [
    {
      "spotify.play": {
        "artist": [
          "Taylor Swift"
        ],
        "d

In [22]:
import re, json

def extract_all_calls(text):
    matches = re.findall(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    calls = []
    for m in matches:
        try: calls.append(json.loads(m.strip()))
        except json.JSONDecodeError: pass
    return calls

def score_parallel(calls, gt_list):
    if len(calls) != len(gt_list):
        return 0.0  # wrong number of calls emitted
    matched = [False] * len(gt_list)
    total_score = 0
    for call in calls:
        fn_name = call.get("name")
        args = call.get("arguments", {})
        best = 0
        best_idx = -1
        for i, gt in enumerate(gt_list):
            if matched[i]: continue
            if fn_name not in gt: continue
            gt_params = gt[fn_name]
            s = sum(1 for p, allowed in gt_params.items()
                     if str(args.get(p, "")) in [str(a) for a in allowed])
            s = s / len(gt_params) if gt_params else 0
            if s > best:
                best, best_idx = s, i
        if best_idx >= 0:
            matched[best_idx] = True
            total_score += best
    return total_score / len(gt_list)

In [23]:
def build_prompt(q):
    fn = q["function"][0]
    user_msg = q["question"][0][0]["content"]
    return [{"role": "user", "content": (
        f"Available function:\n{json.dumps(fn, indent=2)}\n\n"
        f"User: {user_msg}\n"
        f"This may require multiple calls. Wrap each in <tool_call></tool_call>, "
        f'format: {{"name": ..., "arguments": {{...}}}}'
    )}]

results = []
for q in p_questions:
    prompt = build_prompt(q)
    inputs = tokenizer.apply_chat_template(prompt, tokenize=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False).to(model.device)
    out = model.generate(inputs, max_new_tokens=256)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    calls = extract_all_calls(text)
    gt = gt_by_id_parallel[q["id"]]
    score = score_parallel(calls, gt)
    results.append(score)
    print(q["id"], "score:", score)

print("Average:", sum(results)/len(results))

parallel_0 score: 1.0
parallel_1 score: 1.0
parallel_2 score: 1.0
parallel_3 score: 1.0
parallel_4 score: 1.0
parallel_5 score: 0.0
parallel_6 score: 1.0
parallel_7 score: 0.0
parallel_8 score: 0.0
parallel_9 score: 0.0
parallel_10 score: 1.0
parallel_11 score: 1.0
parallel_12 score: 1.0
parallel_13 score: 1.0
parallel_14 score: 1.0
Average: 0.7333333333333333


In [24]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B", max_seq_length=2048,
    load_in_4bit=True, fast_inference=False,
)
FastLanguageModel.for_inference(base_model)

base_results = []
for q in p_questions:
    prompt = build_prompt(q)
    inputs = tokenizer.apply_chat_template(prompt, tokenize=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False).to(base_model.device)
    out = base_model.generate(inputs, max_new_tokens=256)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    calls = extract_all_calls(text)
    gt = gt_by_id_parallel[q["id"]]
    score = score_parallel(calls, gt)
    base_results.append(score)
    print(q["id"], "score:", score)

print("Base Average:", sum(base_results)/len(base_results))

==((====))==  Unsloth 2026.7.6: Fast Qwen3_5 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

parallel_0 score: 0.0
parallel_1 score: 1.0
parallel_2 score: 1.0
parallel_3 score: 0.0
parallel_4 score: 0.0
parallel_5 score: 0.0
parallel_6 score: 1.0
parallel_7 score: 0.0
parallel_8 score: 0.0
parallel_9 score: 0.0
parallel_10 score: 1.0
parallel_11 score: 0.0
parallel_12 score: 0.0
parallel_13 score: 0.0
parallel_14 score: 1.0
Base Average: 0.3333333333333333


### Result

On the held-out `parallel` category (multi-call prompts, never seen in training):

| Model | Avg. score (n=15) |
|---|---|
| Base Qwen3.5-4B | 0.33 |
| Fine-tuned (FunRL-style GRPO) | 0.73 |

More than double, on genuinely unseen data — suggests the training generalized to a harder task rather than just memorizing training examples. Small sample size, custom scorer — a directional result, not a formal benchmark claim.